##### ARTI 560 - Computer Vision

## Visual Representations with DINOv2 - Exercise

### Exercise 1: Unsupervised Clustering

In this exercise, you will use the `KMeans` algorithm from sklearn to group 20 images from the Oxford Pet dataset into 2 clusters (Cats vs. Dogs) based purely on their CLS tokens.

Instructions:

1.  Extract the 384-dimensional [CLS] tokens from 20 images of the Oxford-IIIT Pet dataset. Ensure your selection includes a mix of both cats and dogs.

2. Apply K-Means Clustering ($n=2$) to group the vectors based on mathematical similarity rather than provided labels.

3. Compare the predicted clusters against ground-truth labels.

In [ ]:
from pathlib import Path
import random
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from torchvision.datasets import OxfordIIITPet
from transformers import AutoImageProcessor, AutoModel

MODEL_ID = "facebook/dinov2-small"
DATA_ROOT = Path("data/oxford_pet")
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).to(device)
model.eval()

# Download the Oxford-IIIT Pet dataset on first run.
_ = OxfordIIITPet(root=str(DATA_ROOT), split="trainval", download=True)

cat_breeds = {
    "Abyssinian",
    "Bengal",
    "Birman",
    "Bombay",
    "British_Shorthair",
    "Egyptian_Mau",
    "Maine_Coon",
    "Persian",
    "Ragdoll",
    "Russian_Blue",
    "Siamese",
    "Sphynx",
}


def breed_from_path(path):
    return re.sub(r"_\d+$", "", path.stem)


def species_from_path(path):
    return "cat" if breed_from_path(path) in cat_breeds else "dog"


@torch.no_grad()
def get_cls_embedding(path):
    image = Image.open(path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)
    outputs = model(**inputs)
    cls_token = outputs.last_hidden_state[:, 0]
    return F.normalize(cls_token, p=2, dim=1).squeeze(0).cpu().numpy()


image_paths = sorted(DATA_ROOT.rglob("*.jpg"))
cat_paths = [path for path in image_paths if species_from_path(path) == "cat"]
dog_paths = [path for path in image_paths if species_from_path(path) == "dog"]

rng = random.Random(560)
selected_cats = sorted(rng.sample(cat_paths, 10))
selected_dogs = sorted(rng.sample(dog_paths, 10))
selected_paths = [path for pair in zip(selected_cats, selected_dogs) for path in pair]

ground_truth = np.array([0 if species_from_path(path) == "cat" else 1 for path in selected_paths])
ground_truth_names = np.where(ground_truth == 0, "cat", "dog")

embeddings = np.vstack([get_cls_embedding(path) for path in selected_paths])
print(f"Model: {MODEL_ID}")
print(f"Device: {device}")
print(f"Selected images: {len(selected_paths)}")
print(f"CLS embedding shape: {embeddings.shape}")

kmeans = KMeans(n_clusters=2, random_state=560, n_init=20)
clusters = kmeans.fit_predict(embeddings)

# K-Means cluster ids are arbitrary, so we test both label alignments.
direct_accuracy = (clusters == ground_truth).mean()
flipped_accuracy = ((1 - clusters) == ground_truth).mean()

if flipped_accuracy > direct_accuracy:
    aligned_predictions = 1 - clusters
    accuracy = flipped_accuracy
else:
    aligned_predictions = clusters
    accuracy = direct_accuracy

predicted_species = np.where(aligned_predictions == 0, "cat", "dog")

results = pd.DataFrame(
    {
        "image": [path.name for path in selected_paths],
        "breed": [breed_from_path(path) for path in selected_paths],
        "ground_truth": ground_truth_names,
        "cluster_id": clusters,
        "predicted_species": predicted_species,
    }
)

display(results)
print("\nGround-truth vs. predicted species:")
display(pd.crosstab(results["ground_truth"], results["predicted_species"]))
print(f"\nClustering accuracy after label alignment: {accuracy:.2%}")

projection = PCA(n_components=2, random_state=560).fit_transform(embeddings)
plt.figure(figsize=(8, 6))
colors = {"cat": "tab:blue", "dog": "tab:orange"}

for species in ["cat", "dog"]:
    mask = results["predicted_species"] == species
    plt.scatter(
        projection[mask, 0],
        projection[mask, 1],
        label=f"Predicted {species}",
        s=70,
        c=colors[species],
        alpha=0.8,
    )

for i, path in enumerate(selected_paths):
    plt.text(projection[i, 0] + 0.01, projection[i, 1] + 0.01, path.stem, fontsize=7)

plt.title("K-Means Clustering on DINOv2 CLS Tokens")
plt.xlabel("PCA component 1")
plt.ylabel("PCA component 2")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

### Exercise 2: Image Classification with DINOv2

In this exercise you'll use a DINOv2 model with a pre-trained linear head to classify an image. You will observe how the model maps visual features to specific ImageNet-1k categories.

Instructions:
1. For this exercise, you must use the following Model ID. This specific checkpoint includes the necessary classification head trained on ImageNet-1k:

    Model ID: `facebook/dinov2-small-imagenet1k-1-layer`

2. Find an image online to make the inference. To ensure the model has a fair chance of success, the image should belong to one of the ImageNet-1k classes (e.g., a Golden Retriever, a grand piano, a school bus, or a coffee mug).

In [ ]:
from io import BytesIO
from pathlib import Path
from urllib.request import urlopen

import torch.nn.functional as F
from transformers import AutoImageProcessor, AutoModelForImageClassification

MODEL_ID = "facebook/dinov2-small-imagenet1k-1-layer"
IMAGE_SOURCE = "https://github.com/pytorch/hub/raw/master/images/dog.jpg"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageClassification.from_pretrained(MODEL_ID).to(device)
model.eval()


def load_image(source):
    source = str(source)
    if source.startswith(("http://", "https://")):
        with urlopen(source) as response:
            return Image.open(BytesIO(response.read())).convert("RGB")
    return Image.open(Path(source)).convert("RGB")


img = load_image(IMAGE_SOURCE)
inputs = processor(images=img, return_tensors="pt").to(device)

with torch.no_grad():
    backbone_outputs = model.dinov2(**inputs)
    cls_embedding = backbone_outputs.last_hidden_state[:, 0]
    cls_embedding = F.normalize(cls_embedding, p=2, dim=1)

    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)
    top_probs, top_indices = torch.topk(probs, k=5, dim=-1)

print(f"Model: {MODEL_ID}")
print(f"Device: {device}")
print(f"Image source: {IMAGE_SOURCE}")
print(f"CLS embedding shape: {tuple(cls_embedding.shape)}")
print(f"First 10 embedding values: {cls_embedding[0, :10].cpu().tolist()}\n")
print("Top-5 ImageNet-1k predictions:")

for rank, (idx, score) in enumerate(zip(top_indices[0].tolist(), top_probs[0].tolist()), start=1):
    label = model.config.id2label[idx]
    print(f"{rank}. {label}: {score:.4f} ({score * 100:.2f}%)")

top_idx = top_indices[0, 0].item()
top_label = model.config.id2label[top_idx]
top_score = top_probs[0, 0].item()

plt.figure(figsize=(8, 8))
plt.imshow(img)
plt.title(f"Top-1 prediction: {top_label} ({top_score * 100:.1f}%)")
plt.axis("off")
plt.show()